# 02 - Generate Pseudo Labels

This notebook pseudo-labels unlabeled dashcam videos with a pretrained YOLOPv2 ONNX teacher model on CPU.

Paper reference implemented in this notebook:
- Pseudo-labeling for scalable 3D object detection (Caine et al., arXiv 2021).

## Section 1 - Setup
Install dependencies and import local pipeline wrappers.

In [ ]:
%pip install -q onnxruntime opencv-python pandas numpy tqdm pillow

In [ ]:
from pathlib import Path
import sys
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

repo_root = Path.cwd().resolve().parents[0] if (Path.cwd() / 'notebooks').exists() else Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from src.pipeline.detector import YOLOPv2
from src.pipeline.preprocessor import preprocess

## Section 2 - Paths and Configuration
Define source videos, output folders, and pseudo-label quality thresholds.

In [ ]:
video_dir = repo_root / 'data' / 'dashcam'
data_root = repo_root / 'data' / 'pseudo'
images_dir = data_root / 'images'
labels_dir = data_root / 'labels'
masks_dir = data_root / 'masks'
for p in (images_dir, labels_dir, masks_dir):
    p.mkdir(parents=True, exist_ok=True)

model_path = repo_root / 'models' / 'yolopv2.onnx'
manifest_path = data_root / 'manifest.csv'
target_fps = 2
conf_thresh = 0.70
min_detections = 2
domain_label = 1

print(f'Video input: {video_dir}')
print(f'Model path: {model_path}')

## Section 3 - Helpers
Convert frame sampling and model outputs into YOLO labels and segmentation masks.

In [ ]:
def iter_frames(video_path: Path, fps: int):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        cap.release()
        return

    src_fps = cap.get(cv2.CAP_PROP_FPS)
    if src_fps <= 0:
        src_fps = float(fps)
    step = max(int(round(src_fps / fps)), 1)

    idx = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if idx % step == 0:
            yield idx, frame
        idx += 1

    cap.release()


def write_yolo_labels(detections, out_path: Path, img_w: int, img_h: int):
    lines = []
    for det in detections:
        cls_id = int(det['cls'])
        x1, y1, x2, y2 = det['bbox']
        w = max(float(x2 - x1), 1.0)
        h = max(float(y2 - y1), 1.0)
        cx = (float(x1) + w * 0.5) / img_w
        cy = (float(y1) + h * 0.5) / img_h
        lines.append(f'{cls_id} {cx:.6f} {cy:.6f} {w / img_w:.6f} {h / img_h:.6f}')
    out_path.write_text('\n'.join(lines), encoding='utf-8')


def parse_teacher_outputs(output):
    # Supports current project detector output style; adapt if model output format changes.
    detections = output.get('detections', [])
    lane_mask = output.get('lane_mask')
    drivable_mask = output.get('drivable_mask')
    return detections, lane_mask, drivable_mask

## Section 4 - Pseudo-Label Generation
Run the teacher model, keep confident frames, and save labels/masks for training.

In [ ]:
if not model_path.exists():
    raise FileNotFoundError(f'Expected teacher ONNX at {model_path}')
if not video_dir.exists():
    raise FileNotFoundError(f'Dashcam folder missing: {video_dir}')

teacher = YOLOPv2(str(model_path))
rows = []
video_files = sorted(video_dir.glob('*.mp4')) + sorted(video_dir.glob('*.mov')) + sorted(video_dir.glob('*.avi'))

for video_path in tqdm(video_files, desc='Videos'):
    for frame_idx, frame in iter_frames(video_path, target_fps):
        inp = preprocess(frame)
        pred = teacher(inp)
        detections, lane_mask, drivable_mask = parse_teacher_outputs(pred)

        # Pseudo-label filtering from Caine et al. (arXiv 2021).
        detections = [d for d in detections if float(d.get('conf', 0.0)) >= conf_thresh]
        if len(detections) < min_detections:
            continue

        stem = f'{video_path.stem}_{frame_idx:06d}'
        img_path = images_dir / f'frame_{stem}.jpg'
        label_path = labels_dir / f'frame_{stem}.txt'
        lane_path = masks_dir / f'lane_{stem}.png'
        drivable_path = masks_dir / f'drivable_{stem}.png'

        cv2.imwrite(str(img_path), frame)

        if lane_mask is None:
            lane_mask = np.zeros(frame.shape[:2], dtype=np.uint8)
        if drivable_mask is None:
            drivable_mask = np.zeros(frame.shape[:2], dtype=np.uint8)

        cv2.imwrite(str(lane_path), (lane_mask > 0).astype(np.uint8) * 255)
        cv2.imwrite(str(drivable_path), (drivable_mask > 0).astype(np.uint8) * 255)
        write_yolo_labels(detections, label_path, frame.shape[1], frame.shape[0])

        rows.append({
            'image_path': str(img_path),
            'label_path': str(label_path),
            'lane_mask_path': str(lane_path),
            'drivable_mask_path': str(drivable_path),
            'domain': domain_label,
            'source_video': video_path.name,
        })

print(f'Collected pseudo-labeled frames: {len(rows)}')

## Section 5 - Manifest Export
Save dataset index for mixed-domain finetuning (`domain=1` for real dashcam).

In [ ]:
pd.DataFrame(rows).to_csv(manifest_path, index=False)
print(f'Manifest saved to {manifest_path}')